# 10 · Measure the gradient cosines  ⚠️ different kernel**This is the only notebook that needs torch, rdkit and `unimol_tools`.**Run it with the `aim_gnn` kernel; run every other notebook with the base kernel.Loading torch's OpenMP runtime beside matplotlib's crashes on this machine, sothis notebook does **no plotting** — it writes a cache that notebook 05 reads.`train.py` never logged `cos(g_i, g_j)`, so the only way to get it is to reloada checkpoint and recompute: one forward + N backward passes per batch. This is**not** re-training — minutes, not hours — but it is real compute.Which checkpoint: **LS** for geometry (isolates the backbone), **AIM-Matrix**for `w_ij` (τ and cos must come from the same model state).

In [ ]:
import sys, json, csv
from pathlib import Path
import numpy as np
sys.path.insert(0, ".")
import common as C

TAG = "11task"          # "2task" or "11task"
METHOD = "ls"           # "ls" | "pcgrad" | "aim_scalar" | "aim_matrix"
SPLIT = "primary"       # primary | guidance | val | test
N_BATCHES = 16
BATCH_SIZE = 32
DEVICE = "cpu"
BACKBONES = ["GNN", "Uni-Mol"]          # GNN first: it is far quicker
DATA_ROOT = C.REPO / "data" / "qm9"

SRC = {"Uni-Mol": C.REPO / "Unimol" / "src", "GNN": C.REPO / "GNN" / "src"}
print("cache will be:", C.cache_path(TAG, METHOD, SPLIT))

In [ ]:
import torch, torch.nn.functional as F

def import_backbone(src: Path):
    for m in ("data", "model", "train", "metrics", "baselines",
              "aim_optimizer", "analysis", "gnn_collate", "unimol_collate"):
        sys.modules.pop(m, None)
    sys.path[:] = [p for p in sys.path if "GNN" not in p and "Unimol" not in p]
    sys.path.insert(0, str(src))
    import importlib
    return importlib.import_module("data"), importlib.import_module("model")

def measure(backbone):
    rd = C.RESULTS[backbone][TAG]
    ck = Path(rd) / f"{METHOD}_n{C.N_TRAIN}_seed{C.SEED}" / "best_model.pt"
    if not ck.exists():
        print(f"  !! no checkpoint {ck}"); return None
    data, model_mod = import_backbone(SRC[backbone])
    cols = data.parse_task_spec(None if TAG == "11task" else ["mu", "eps_LUMO"])
    names = [data.TASK_NAMES[i] for i in cols]; n = len(cols)

    dev = torch.device(DEVICE)
    loaders = data.get_loaders(root=str(DATA_ROOT), n_train=C.N_TRAIN,
                               batch_size=BATCH_SIZE, seed=C.SEED, target_cols=cols)
    model = model_mod.build_model(device=dev, n_tasks=n, head_hidden=64,
                                  trainable_layers=-1)
    ckpt = torch.load(ck, map_location=dev, weights_only=False)
    model.load_state_dict(ckpt["model"]); model.eval()   # autograd still active
    means, stds = loaders["means"].to(dev), loaders["stds"].to(dev)
    shared = model.shared_params()
    d = sum(p.numel() for p in shared)
    print(f"[{backbone}] epoch {ckpt.get('epoch','?')}  d={d:,}  "
          f"(~{n*d*4/1e9:.2f} GB for {n} fp32 gradients)")

    mats = []
    for b, batch in enumerate(loaders[f"{SPLIT}_loader"]):
        if b >= N_BATCHES: break
        batch = data.batch_to_device(batch, dev)
        y = (batch["targets"] - means) / stds
        h = model.shared_forward(batch)
        rows = []
        for i in range(n):
            li = F.l1_loss(model.task_forward(h, i), y[:, i])
            gi = torch.autograd.grad(li, shared, retain_graph=(i < n - 1),
                                     allow_unused=True)
            flat = torch.cat([(g if g is not None else torch.zeros_like(p)).reshape(-1)
                              for g, p in zip(gi, shared)]).float()
            rows.append(F.normalize(flat, dim=0, eps=1e-12))
        G = torch.stack(rows); mats.append((G @ G.t()).cpu().numpy())
        del G, rows, h
        print(f"    batch {b+1}/{N_BATCHES}", end="\r")

    pol = ckpt.get("policy") or {}
    t = pol.get("tau")
    tau = None
    if t is not None:
        a = np.asarray(t.detach().cpu().numpy(), dtype=float)
        tau = np.full((n, n), float(a)) if a.ndim == 0 else a
    return dict(name=backbone, cos=np.stack(mats), tasks=names, tau=tau,
                epoch=int(ckpt.get("epoch", -1)))

In [ ]:
results = [r for r in (measure(b) for b in BACKBONES) if r is not None]

out = C.cache_path(TAG, METHOD, SPLIT)
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["backbone", "batch", "epoch", "tasks", "matrix", "tau"])
    for r in results:
        tk = "|".join(r["tasks"])
        tau_s = "" if r["tau"] is None else "|".join(f"{v:.6f}" for v in np.ravel(r["tau"]))
        for b, M in enumerate(r["cos"]):
            w.writerow([r["name"], b, r["epoch"], tk,
                        "|".join(f"{v:.6f}" for v in M.ravel()), tau_s])
print("\nsaved", out)
print("Now switch to the base kernel and run 05_gradient_geometry.ipynb.")